# CerebroSpheroid V5: Kaggle 2x Tesla T4 DDP Training

Launches multi-GPU `DistributedDataParallel` training on Kaggle `NvidiaTeslaT4` (`2 x 15 GB`) with:
- **Hardware-Native FP16 AMP + GradScaler** on `sm_75` Tesla T4 Tensor Cores (avoids slow software BF16 emulation)
- **Dual-GPU `torchrun` DDP** with `find_unused_parameters=True`, gradient accumulation `no_sync()`, and rank-distinct data sampling
- **Continuous `bioflo.in` Cloud Sync** (`from_scratch=False` resumes from the latest synced `v5_small` checkpoint automatically)
- **Wall-Clock Guard (`--max-hours 8.5`)** to flush checkpoints cleanly before Kaggle session timeout

In [ ]:
import subprocess
import sys
import torch

print(f"Python  : {sys.version.split()[0]}")
print(f"PyTorch : {torch.__version__}")
NUM_GPUS = torch.cuda.device_count() if torch.cuda.is_available() else 0
print(f"CUDA    : {torch.cuda.is_available()} (devices={NUM_GPUS})")

if NUM_GPUS > 0:
    for idx in range(NUM_GPUS):
        props = torch.cuda.get_device_properties(idx)
        print(
            f"  [GPU {idx}] {props.name} | sm_{props.major}{props.minor} | "
            f"{props.total_memory / (1024**3):.2f} GB VRAM"
        )
    subprocess.run(["nvidia-smi"], check=False)
else:
    print("WARNING: No CUDA GPU detected.")

# Optional: Authenticate HuggingFace API if HF_TOKEN is in Kaggle Secrets
try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    hf_token = user_secrets.get_secret("HF_TOKEN")
    if hf_token:
        import os
        os.environ["HF_TOKEN"] = hf_token
        print("✓ Successfully loaded HF_TOKEN from Kaggle Secrets")
except Exception:
    pass


In [ ]:
import importlib.util
import os
import subprocess
import sys

REPO_URL = "https://github.com/AjitKumar-git/cerebro_spheroid_v5.git"
REPO_DIR = "/tmp/cerebro_spheroid_v5"

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "--depth", "1", "origin", "main"], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "reset", "--hard", "origin/main"], check=True)

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

commit = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip()
print(f"Repository synced at commit [{commit}] in {REPO_DIR}")
assert os.path.exists(os.path.join(REPO_DIR, "data", "tokenizer")), "Missing offline LLaMA-32k tokenizer"

missing_pkgs = [
    pkg for pkg in ("transformers", "datasets", "accelerate", "sentencepiece", "pyarrow")
    if importlib.util.find_spec(pkg) is None
]
if missing_pkgs:
    print(f"Installing missing packages (keeping preinstalled PyTorch): {missing_pkgs}")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing_pkgs], check=True)
else:
    print("All required non-PyTorch packages are already present.")

In [ ]:
import os
from src.cloud_sync import check_remote_manifest

MODEL_PRESET = "v5_small"
MIX = "auto"
TOTAL_STEPS = 1_000_000
BATCH_SIZE = 8
TARGET_BATCH_SIZE = 64
PROMPT_LEN = 32
SEQ_LEN = 512
MAX_LR = "5e-4"
WARMUP_STEPS = 500
SAVE_INTERVAL = 1000
LOG_INTERVAL = 10
EVAL_INTERVAL = 250
CLOUD_SYNC_INTERVAL = 100
MAX_HOURS = 8.5
# Mixed precision: Hardware-Native FP16 on Tesla T4 Tensor Cores
AMP_DTYPE = "float16"
# Deliberation mode: "teacher_forcing" (15,000+ tok/s parallel) or "scheduled" (autoregressive)
DELIBERATION_MODE = "teacher_forcing"
SCHEDULE_STEPS = 0
MIN_SAMPLING_P = 1.00
PITFALL_PROB = 0.0
LOSS_CEILING = 3.25
LOSS_FLOOR = 2.80
CHECKPOINT_DIR = f"/kaggle/working/checkpoints/{MODEL_PRESET}"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

active_ranks = max(1, NUM_GPUS)
grad_accum = max(1, (TARGET_BATCH_SIZE + (active_ranks * BATCH_SIZE) - 1) // (active_ranks * BATCH_SIZE))
eff_batch = active_ranks * BATCH_SIZE * grad_accum
tok_per_step = eff_batch * (PROMPT_LEN + SEQ_LEN)
print(f"Active GPUs     : {active_ranks}")
print(f"Effective Batch : {eff_batch} prompts/step (accum={grad_accum}) | ({PROMPT_LEN}, {SEQ_LEN}) = {tok_per_step:,} tok/step")

manifest = check_remote_manifest(model=MODEL_PRESET)
print(f"Cloud Hub Status: {manifest}")


In [ ]:
import os
import shutil
import subprocess
import sys

train_args = [
    "scripts/train_v5_graded_curriculum.py",
    "--model-preset", MODEL_PRESET,
    "--mix", MIX,
    "--total-steps", str(TOTAL_STEPS),
    "--batch-size", str(BATCH_SIZE),
    "--target-batch-size", str(TARGET_BATCH_SIZE),
    "--prompt-len", str(PROMPT_LEN),
    "--seq-len", str(SEQ_LEN),
    "--no-elastic-window",
    "--amp-dtype", AMP_DTYPE,
    "--max-lr", str(MAX_LR),
    "--warmup-steps", str(WARMUP_STEPS),
    "--resume-warmup-steps", "100",
    "--log-interval", str(LOG_INTERVAL),
    "--eval-interval", str(EVAL_INTERVAL),
    "--save-interval", str(SAVE_INTERVAL),
    "--cloud-sync",
    "--cloud-sync-interval", str(CLOUD_SYNC_INTERVAL),
    "--deliberation-mode", DELIBERATION_MODE,
    "--schedule-steps", str(SCHEDULE_STEPS),
    "--min-sampling-p", str(MIN_SAMPLING_P),
    "--pitfall-prob", str(PITFALL_PROB),
    "--loss-window-governor",
    "--loss-ceiling", str(LOSS_CEILING),
    "--loss-floor", str(LOSS_FLOOR),
    "--loss-lr-damping",
    "--optimizer", "spheroid",
    "--checkpoint-dir", CHECKPOINT_DIR,
    "--max-hours", str(MAX_HOURS),
]

if NUM_GPUS > 1:
    torchrun_bin = shutil.which("torchrun")
    launcher = [torchrun_bin] if torchrun_bin else [sys.executable, "-m", "torch.distributed.run"]
    cmd = [*launcher, "--standalone", f"--nproc_per_node={NUM_GPUS}", *train_args]
else:
    cmd = [sys.executable, *train_args]

env = os.environ.copy()
env["OMP_NUM_THREADS"] = "2"
env["MALLOC_ARENA_MAX"] = "2"
env["TOKENIZERS_PARALLELISM"] = "false"
env["HF_HOME"] = "/tmp/hf_cache"
env["NCCL_IB_DISABLE"] = "1"
env["NCCL_DEBUG"] = "WARN"
env["PYTHONUNBUFFERED"] = "1"

print("Launching:", " ".join(cmd))
subprocess.run(cmd, env=env, check=True)


In [ ]:
import json
import os
import subprocess
import sys

eval_out = "/kaggle/working/eval_results"
os.makedirs(eval_out, exist_ok=True)
candidates = [
    os.path.join(CHECKPOINT_DIR, f"{MODEL_PRESET}_latest.pt"),
    os.path.join(CHECKPOINT_DIR, f"{MODEL_PRESET}_best.pt"),
]
ckpt_to_eval = next((p for p in candidates if os.path.exists(p)), None)
if ckpt_to_eval:
    eval_device = "cuda:0" if NUM_GPUS > 0 else "cpu"
    subprocess.run(
        [
            sys.executable,
            "eval/run_benchmark.py",
            "--checkpoint", ckpt_to_eval,
            "--samples", "50",
            "--device", eval_device,
            "--output-dir", eval_out,
        ],
        check=True,
    )
    report_file = os.path.join(eval_out, "benchmark_report.json")
    if os.path.exists(report_file):
        with open(report_file) as f:
            print(json.dumps(json.load(f), indent=2)[:2000])
else:
    print("No checkpoint found to evaluate.")